# Model Architecture Visualization세 모델의 입출력 shape 과 블록 구조를 동일 포맷으로 시각화한다.1. **5채널 단일 인코더** — `baseline.baseline_model.AdvancedBaselineModel`   - 입력 (B, 5, 5000) — EMG 2채널 + IMU 3채널 동기화 (1000Hz)2. **Multimodal intermediate fusion** — `Multimodal.mm_model.InterFusionClassifier`   - 입력 (B, 2, 5000) + (B, 3, 500) — EMG/IMU 분리 입력3. **Pretrained Multimodal** — `pretrain_Multimodal.pretrain_model.PretrainInterFusionClassifier`   - Phase 1 SSL pretraining + Phase 2 fine-tune. 구조는 MM 과 동일, encoder weight 만 SSL 로부터 초기화.

In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# 프로젝트 루트 추가
sys.path.append(os.path.abspath('..'))

from baseline.baseline_model import AdvancedBaselineModel
from Multimodal.mm_model import InterFusionClassifier
from pretrain_Multimodal.pretrain_model import (
    PretrainInterFusionClassifier, EMGSSLModel, IMUSSLModel
)

# === 모델 입출력 sanity check ===
m_5ch  = AdvancedBaselineModel(in_channels=5, num_classes=10)
m_mm   = InterFusionClassifier(num_classes=10)
m_ptmm = PretrainInterFusionClassifier(num_classes=10)

x_5ch = torch.randn(4, 5, 5000)
x_emg = torch.randn(4, 2, 5000)
x_imu = torch.randn(4, 3,  500)

out_5ch = m_5ch(x_5ch)
log_mm,  feat_mm  = m_mm (x_emg, x_imu)
log_ptm, feat_ptm = m_ptmm(x_emg, x_imu)

print('=== Forward shape check ===')
print(f'5ch    : in {str(tuple(x_5ch.shape)):>22}  ->  out {tuple(out_5ch.shape)}')
print(f'MM     : in {str(tuple(x_emg.shape))} + {str(tuple(x_imu.shape))}  '
      f'->  out {tuple(log_mm.shape)}, feat {tuple(feat_mm.shape)}')
print(f'pt-MM  : in {str(tuple(x_emg.shape))} + {str(tuple(x_imu.shape))}  '
      f'->  out {tuple(log_ptm.shape)}, feat {tuple(feat_ptm.shape)}')

for name, m in [('5ch', m_5ch), ('MM', m_mm), ('pt-MM', m_ptmm)]:
    p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  params {name:>6}: {p:>12,}')

## 1. 시각화 헬퍼

In [ ]:
# 블록 색깔 규약
COLOR = {
    'io':       '#FFF4C2',  # 입출력
    'stem':     '#FFB8B8',  # Conv1d stem
    'tcn':      '#B8DAFF',  # ResidualTCNBlock
    'pool':     '#DDDDDD',  # MaxPool / AdaptiveAvgPool
    'fusion':   '#D8B4FE',  # concat + fusion
    'interp':   '#FED8B8',  # interpolate
    'cls':      '#BBE7BB',  # classifier / projection head
    'ssl':      '#FBBBE3',  # SSL projection head
}


def add_block(ax, x, y, w, h, text, kind='tcn', fontsize=9):
    rect = FancyBboxPatch((x - w/2, y - h/2), w, h,
                          boxstyle='round,pad=0.02',
                          linewidth=1.2, edgecolor='black',
                          facecolor=COLOR[kind])
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize)


def add_arrow(ax, x1, y1, x2, y2, shape=None, side='right'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', lw=1.2, color='dimgray'))
    if shape:
        # 화살표 옆에 shape 라벨
        dx = 0.18 if side == 'right' else -0.18
        ha = 'left' if side == 'right' else 'right'
        ax.text((x1 + x2)/2 + dx, (y1 + y2)/2, shape,
                fontsize=8, color='#2c5b9c', family='monospace',
                ha=ha, va='center')


def vertical_diagram(ax, blocks, x=0, w=4.5, h=0.55, gap=0.35, title=''):
    """blocks: list of (text, kind, shape_after)."""
    y = 0
    centers = []
    for txt, kind, _ in blocks:
        add_block(ax, x, y, w, h, txt, kind)
        centers.append(y)
        y -= (h + gap)
    # 화살표
    for i in range(len(blocks) - 1):
        y_from = centers[i] - h/2
        y_to   = centers[i+1] + h/2
        shape  = blocks[i][2]
        add_arrow(ax, x, y_from, x, y_to, shape=shape)
    if title:
        ax.set_title(title, fontsize=11, pad=10)
    return centers


def style_axes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_aspect('equal')


print('helpers ready.')

## 2. 5채널 단일 인코더 (`AdvancedBaselineModel`)Input  : `(B, 5, 5000)` — EMG 2 + IMU 3 동기화Output : `(B, 10)` — class logits

In [ ]:
blocks_5ch = [
    ('Input  (B, 5, 5000)',              'io',  '(B, 5, 5000)'),
    ('Stem  Conv1d(5→64, k=11, s=5) + BN + GELU', 'stem', '(B, 64, 1000)'),
    ('ResTCN(64→64,   d=1)',             'tcn', '(B, 64, 1000)'),
    ('ResTCN(64→128,  d=2)',             'tcn', '(B, 128, 1000)'),
    ('MaxPool 2',                         'pool', '(B, 128, 500)'),
    ('ResTCN(128→128, d=4)',             'tcn', '(B, 128, 500)'),
    ('ResTCN(128→256, d=8)',             'tcn', '(B, 256, 500)'),
    ('MaxPool 2',                         'pool', '(B, 256, 250)'),
    ('ResTCN(256→256, d=16)',            'tcn', '(B, 256, 250)'),
    ('ResTCN(256→512, d=32)',            'tcn', '(B, 512, 250)'),
    ('AdaptiveAvgPool(1) + squeeze',     'pool', '(B, 512)'),
    ('Linear 512→256 + GELU + Dropout',  'cls', '(B, 256)'),
    ('Linear 256→10',                    'cls', '(B, 10)'),
    ('Output  (B, 10)',                  'io',  ''),
]

fig, ax = plt.subplots(figsize=(7, 12))
vertical_diagram(ax, blocks_5ch, x=0, w=5.0, h=0.55, gap=0.35,
                 title='5채널 단일 인코더 — AdvancedBaselineModel')
style_axes(ax)
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(min(0, -1 * (len(blocks_5ch) * 0.95)) - 0.5, 0.6)

# 범례
from matplotlib.patches import Patch
legend = [Patch(facecolor=COLOR[k], edgecolor='black', label=l)
          for k, l in [('io', 'I/O'), ('stem', 'Conv stem'),
                       ('tcn', 'ResidualTCNBlock'), ('pool', 'Pool'),
                       ('cls', 'Classifier')]]
ax.legend(handles=legend, loc='lower right', fontsize=8, frameon=True)
plt.tight_layout(); plt.show()

## 3. Multimodal Intermediate Fusion (`InterFusionClassifier`)Input  : `(B, 2, 5000)` EMG + `(B, 3, 500)` IMUOutput : `(B, 10)` class logits + `(B, 512)` fused feature핵심: 각 인코더의 AdaptiveAvgPool **이전** temporal feature map 을 꺼내 시간축 정렬 후 concat → joint TCN.

In [ ]:
def draw_mm_diagram(ax, title='Multimodal Intermediate Fusion — InterFusionClassifier',
                    show_phase1=False):
    """두 갈래 (EMG/IMU) → concat → fusion → classifier."""
    # 좌측 EMG, 우측 IMU
    x_emg, x_imu, x_mid = -3.0, 3.0, 0.0
    w, h, gap = 4.2, 0.55, 0.32

    emg_branch = [
        ('EMG Input  (B, 2, 5000)',                       'io',   '(B, 2, 5000)'),
        ('EMG Stem  Conv1d(2→64, k=11, s=5)',             'stem', '(B, 64, 1000)'),
        ('ResTCN(64→64, d=1)',                            'tcn',  '(B, 64, 1000)'),
        ('ResTCN(64→128, d=2) + MaxPool 2',               'tcn',  '(B, 128, 500)'),
        ('ResTCN(128→128, d=4)',                          'tcn',  '(B, 128, 500)'),
        ('ResTCN(128→256, d=8)',                          'tcn',  '(B, 256, 500)'),
    ]
    imu_branch = [
        ('IMU Input  (B, 3, 500)',                        'io',   '(B, 3, 500)'),
        ('IMU Stem  Conv1d(3→64, k=11, s=5)',             'stem', '(B, 64, 100)'),
        ('ResTCN(64→64, d=1)',                            'tcn',  '(B, 64, 100)'),
        ('ResTCN(64→128, d=2) + MaxPool 2',               'tcn',  '(B, 128, 50)'),
        ('ResTCN(128→128, d=4)',                          'tcn',  '(B, 128, 50)'),
        ('ResTCN(128→256, d=8)',                          'tcn',  '(B, 256, 50)'),
    ]

    def draw_branch(blocks, x):
        y = 0; cs = []
        for txt, kind, _ in blocks:
            add_block(ax, x, y, w, h, txt, kind, fontsize=8)
            cs.append(y); y -= (h + gap)
        for i in range(len(blocks) - 1):
            add_arrow(ax, x, cs[i]-h/2, x, cs[i+1]+h/2, shape=blocks[i][2])
        return cs

    emg_cs = draw_branch(emg_branch, x_emg)
    imu_cs = draw_branch(imu_branch, x_imu)

    # IMU 쪽에 interpolate 박스 추가
    y_interp = imu_cs[-1] - (h + gap)
    add_block(ax, x_imu, y_interp, w, h, 'Interpolate t→500', 'interp', fontsize=8)
    add_arrow(ax, x_imu, imu_cs[-1]-h/2, x_imu, y_interp+h/2,
              shape='(B, 256, 50)')

    # concat 박스 (중앙)
    y_concat = y_interp - (h + gap*1.8)
    add_block(ax, x_mid, y_concat, w*1.1, h, 'Concat  (channel axis)', 'fusion', fontsize=9)
    add_arrow(ax, x_emg, emg_cs[-1]-h/2, x_mid - w*0.5, y_concat+h/2,
              shape='(B, 256, 500)', side='right')
    add_arrow(ax, x_imu, y_interp-h/2, x_mid + w*0.5, y_concat+h/2,
              shape='(B, 256, 500)', side='left')

    # fusion + classifier (중앙 일렬)
    central = [
        ('Conv1d(512→64, k=7, s=2) + BN + GELU',          'stem',  '(B, 512, 500)'),
        ('ResTCN(64→64,   d=1)',                          'tcn',   '(B, 64, 250)'),
        ('ResTCN(64→128,  d=2)',                          'tcn',   '(B, 64, 250)'),
        ('ResTCN(128→128, d=4)',                          'tcn',   '(B, 128, 250)'),
        ('ResTCN(128→256, d=8)',                          'tcn',   '(B, 128, 250)'),
        ('ResTCN(256→256, d=16)',                         'tcn',   '(B, 256, 250)'),
        ('ResTCN(256→512, d=32)',                         'tcn',   '(B, 256, 250)'),
        ('AdaptiveAvgPool(1) + squeeze',                  'pool',  '(B, 512, 250)'),
        ('Linear 512→256 + GELU + Dropout + Linear 256→10', 'cls', '(B, 512)'),
        ('Output  logits (B, 10) ─── features (B, 512)',  'io',    ''),
    ]
    y = y_concat - (h + gap); cs = []
    for txt, kind, _ in central:
        add_block(ax, x_mid, y, w*1.1, h, txt, kind, fontsize=8)
        cs.append(y); y -= (h + gap)
    # concat → 첫 central
    add_arrow(ax, x_mid, y_concat-h/2, x_mid, cs[0]+h/2, shape='(B, 512, 500)')
    for i in range(len(central) - 1):
        add_arrow(ax, x_mid, cs[i]-h/2, x_mid, cs[i+1]+h/2, shape=central[i][2])

    ax.set_title(title, fontsize=11, pad=10)
    return y


fig, ax = plt.subplots(figsize=(13, 14))
y_end = draw_mm_diagram(ax)
style_axes(ax)
ax.set_xlim(-6, 6)
ax.set_ylim(y_end - 0.8, 0.6)

from matplotlib.patches import Patch
legend = [Patch(facecolor=COLOR[k], edgecolor='black', label=l)
          for k, l in [('io', 'I/O'), ('stem', 'Conv stem'),
                       ('tcn', 'ResTCN'), ('pool', 'Pool'),
                       ('interp', 'Interpolate'), ('fusion', 'Concat/Fusion'),
                       ('cls', 'Classifier')]]
ax.legend(handles=legend, loc='lower right', fontsize=8, frameon=True)
plt.tight_layout(); plt.show()

## 4. Pretrained Multimodal (`PretrainInterFusionClassifier`)Phase 1: 각 모달리티 별 SimCLR 학습 (augmentation 페어 → projection head → NT-Xent loss)         학습 후 encoder weight 만 저장.Phase 2: MM 모델과 동일 구조에 Phase 1 encoder weight 로드 후 end-to-end fine-tune.

In [ ]:
# Phase 1: SSL pretraining diagram (EMG + IMU 병렬)
fig, ax = plt.subplots(figsize=(13, 8))

# 그래프 좌표 (※ 텐서 변수 x_emg/x_imu 와 이름 충돌 피하려고 pos_ 접두사)
pos_emg = -3.5
emg_phase1 = [
    ('EMG Input  (B, 2, 5000)',                     'io',   '(B, 2, 5000)'),
    ('Augmentation (jitter+scale+shift)',           'interp', '(B, 2, 5000) × 2 views'),
    ('EMGEncoder  Stem + 4× ResTCN + AvgPool',      'tcn',  '(B, 256) × 2 views'),
    ('Projection 256→128(+BN+GELU)→64 + L2norm',    'ssl',  '(B, 64) × 2 views'),
    ('NT-Xent contrastive loss',                    'io',   ''),
]

pos_imu = 3.5
imu_phase1 = [
    ('IMU Input  (B, 3, 500)',                      'io',   '(B, 3, 500)'),
    ('Augmentation (jitter+scale+shift+sign_flip)', 'interp', '(B, 3, 500) × 2 views'),
    ('IMUEncoder  Stem + 4× ResTCN + AvgPool',      'tcn',  '(B, 256) × 2 views'),
    ('Projection 256→128(+BN+GELU)→64 + L2norm',    'ssl',  '(B, 64) × 2 views'),
    ('NT-Xent contrastive loss',                    'io',   ''),
]

w, h, gap = 5.0, 0.55, 0.32

def draw_branch(blocks, x):
    y = 0; cs = []
    for txt, kind, _ in blocks:
        add_block(ax, x, y, w, h, txt, kind, fontsize=8)
        cs.append(y); y -= (h + gap)
    for i in range(len(blocks) - 1):
        add_arrow(ax, x, cs[i]-h/2, x, cs[i+1]+h/2, shape=blocks[i][2])
    return cs


emg_cs = draw_branch(emg_phase1, pos_emg)
imu_cs = draw_branch(imu_phase1, pos_imu)

ax.text(0, 0.4, 'Phase 1: per-modality SimCLR pretraining',
        ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(0, -2.2, '↓\n저장: emg_encoder.pth, imu_encoder.pth',
        ha='center', va='center', fontsize=10, color='dimgray')

style_axes(ax)
ax.set_xlim(-7.5, 7.5)
ax.set_ylim(min(emg_cs[-1], imu_cs[-1]) - 1.0, 0.8)

from matplotlib.patches import Patch
legend = [Patch(facecolor=COLOR[k], edgecolor='black', label=l)
          for k, l in [('io', 'I/O / Loss'), ('interp', 'Augmentation'),
                       ('tcn', 'Encoder'), ('ssl', 'Projection head')]]
ax.legend(handles=legend, loc='lower center', ncol=4, fontsize=8, frameon=True)
plt.tight_layout(); plt.show()

In [ ]:
# Phase 2: MM 과 동일 구조, encoder weight 만 Phase 1 에서 로드
fig, ax = plt.subplots(figsize=(13, 14))
y_end = draw_mm_diagram(ax, title='Phase 2: PretrainInterFusionClassifier  (encoders ← Phase 1 weights)')

# 좌상단에 "pretrained" 강조 박스
ax.text(-3.0, 0.6, '* Encoders initialized from\n  Phase 1 SimCLR weights',
        fontsize=9, color='crimson', fontweight='bold', va='bottom')
ax.text( 3.0, 0.6, '* Encoders initialized from\n  Phase 1 SimCLR weights',
        fontsize=9, color='crimson', fontweight='bold', va='bottom')

style_axes(ax)
ax.set_xlim(-6, 6)
ax.set_ylim(y_end - 0.8, 1.2)

from matplotlib.patches import Patch
legend = [Patch(facecolor=COLOR[k], edgecolor='black', label=l)
          for k, l in [('io', 'I/O'), ('stem', 'Conv stem'),
                       ('tcn', 'ResTCN'), ('pool', 'Pool'),
                       ('interp', 'Interpolate'), ('fusion', 'Concat/Fusion'),
                       ('cls', 'Classifier')]]
ax.legend(handles=legend, loc='lower right', fontsize=8, frameon=True)
plt.tight_layout(); plt.show()

## 5. 텍스트 요약 (channels / params / shapes)

In [ ]:
def summarize(name, model, sample_inputs, output_unpacker):
    """단일 forward 후 입력/출력 shape 와 파라미터 수 출력."""
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'─── {name} ───')
    for i, x in enumerate(sample_inputs):
        print(f'  input {i}: {tuple(x.shape)}')
    out = model(*sample_inputs)
    output_unpacker(out)
    print(f'  trainable params: {total:,}')
    # 주요 sub-module 별 파라미터
    sub_counts = {}
    for child_name, child in model.named_children():
        sub_counts[child_name] = sum(p.numel() for p in child.parameters() if p.requires_grad)
    for k, v in sub_counts.items():
        print(f'    {k:<22} {v:>12,}')
    print()


summarize('5채널 단일 인코더', m_5ch, [x_5ch],
          lambda o: print(f'  output: {tuple(o.shape)}  (class logits)'))

summarize('MM (InterFusionClassifier)', m_mm, [x_emg, x_imu],
          lambda o: print(f'  output: logits {tuple(o[0].shape)}, features {tuple(o[1].shape)}'))

summarize('pt-MM (PretrainInterFusionClassifier)', m_ptmm, [x_emg, x_imu],
          lambda o: print(f'  output: logits {tuple(o[0].shape)}, features {tuple(o[1].shape)}'))

# Phase 1 SSL 모델도 별도 출력
print('─── Phase 1: EMGSSLModel / IMUSSLModel ───')
emg_ssl = EMGSSLModel(); imu_ssl = IMUSSLModel()
print(f'  EMGSSLModel: in {tuple(x_emg.shape)}  ->  out {tuple(emg_ssl(x_emg).shape)}')
print(f'  IMUSSLModel: in {tuple(x_imu.shape)}  ->  out {tuple(imu_ssl(x_imu).shape)}')
for name, m in [('EMGSSL', emg_ssl), ('IMUSSL', imu_ssl)]:
    p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  {name:<10} params: {p:,}')